In [ ]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [ ]:
dbutils.widgets.text("catalog_name","projectecommerce","Catalog name")
dbutils.widgets.text("schema_account_name","stdecommercedevcink001", "Storage Account Name")
dbutils.widgets.text("Container_name","raw-ecomm-data-ci", "Container Name")

In [ ]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("schema_account_name")
container_name = dbutils.widgets.get("Container_name")

In [ ]:
# Read CDC stream
df = spark.readStream \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .table(f"{catalog_name}.silver.slv_order_items")

# Filter only required change types
df_union = df.filter("_change_type IN ('insert', 'update_postimage')")

# Transformations
df_union = df_union.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
)

df_union = df_union.withColumn(
    "discount_amount",
    F.ceil(F.col("gross_amount") * (F.col("discount_pct") / 100.0))
)

df_union = df_union.withColumn(
    "sale_amount",
    F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount")
)

df_union = df_union.withColumn(
    "date_id",
    F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType())
)

df_union = df_union.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1)).otherwise(F.lit(0))
)

# Final projection
orders_gold_df = df_union.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sale_amount").alias("net_amount")
)

gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_items/"
print(gold_checkpoint_path)

# Upsert logic
def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog_name}.gold.gld_fact_order_items"

    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)

        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)

        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.transaction_id = batch_table.transaction_id AND gold_table.seq_no = batch_table.seq_no"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

# Streaming write (FIXED)
query = orders_gold_df.writeStream \
    .foreachBatch(upsert_to_gold) \
    .option("checkpointLocation", gold_checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .start()

query.awaitTermination()

# Validation
spark.sql(f"SELECT count(*) FROM {catalog_name}.gold.gld_fact_order_items").show()
spark.sql(f"SELECT max(transaction_date) FROM {catalog_name}.gold.gld_fact_order_items").show()